In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import InMemoryVectorStore
from langchain_groq import ChatGroq

from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel, Field

In [5]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
docs = splitter.split_documents(docs)

embeddings = GoogleGenerativeAIEmbeddings(model = "gemini-embedding-2-preview")
vector_db = InMemoryVectorStore.from_documents(embedding=embeddings, documents=docs)

In [6]:
llm = ChatGroq(model= "openai/gpt-oss-20b")

In [7]:
class RagState(BaseModel):
    question:str = Field(description="user Question")
    documents: list = []
    context:str = Field(description = "context data for user question",default="")
    answer:str = Field(description="final Answer...", default="")

In [8]:
def retrive_node(state:RagState) -> RagState:
    docs = vector_db.similarity_search(query=state.question)
    state.documents = docs
    return state

def create_conntext_node(state: RagState) -> RagState:
    context = ""
    for doc in state.documents:
        context += doc.page_content + "\n\n"
    state.context = context
    return state

def geberate_node(state:RagState) -> RagState:
    prompt = f"""
        You are a assistant and provide the answer for user question based on the provided context.
        If you don't find the relevant answer, then just say 'I dont know.'.
        Context is: {state.context},
        Questions is: {state.question}
    """
    res = llm.invoke(prompt)
    state.answer = res.content
    return state

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver
memory = InMemorySaver()

In [ ]:
graph = StateGraph(RagState)

graph.add_node("retrieve_node", retrieve_node)
graph.add_node("create_context_node", create_context_node)
graph.add_node("generate_node", generate_node)

graph.add_edge(START, 'retrieve_node')
graph.add_edge("retrieve_node", "create_context_node")
graph.add_edge("create_context_node", "generate_node")
graph.add_edge("generate_node", END)

graph = graph.compile()